# 14 · Cost-Sensitive Evaluation

## What this notebook covers
Classification metrics like AUC treat false positives and false negatives as equally costly. In most real applications they are not. A missed fraud case costs $500; a false alert costs $5 in analyst time. This notebook operationalises cost-sensitive evaluation: profit curves, optimal threshold selection, and cost-weighted confusion matrices.

## The cost matrix
A cost matrix C[i][j] specifies the cost of predicting class j when the true class is i:

```
              Predicted
              Neg   Pos
True  Neg   [TN_cost, FP_cost]
      Pos   [FN_cost, TP_cost]
```

For fraud detection: TN_cost=0, FP_cost=-5 (analyst review), FN_cost=-500 (undetected fraud), TP_cost=+50 (fraud caught, partial recovery).

## Optimal threshold selection
The default threshold (0.5) is optimal only when FP and FN costs are equal. With asymmetric costs, the optimal threshold minimises expected cost (or maximises expected profit) over the test set. The profit curve plots expected profit vs threshold — the peak is the operating point.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Imbalanced fraud-like dataset
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=10,
    weights=[0.97, 0.03], random_state=0
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

clf = GradientBoostingClassifier(n_estimators=150, random_state=0)
clf.fit(X_tr, y_tr)
proba_te = clf.predict_proba(X_te)[:, 1]

print(f"Test set: {y_te.sum()} positives / {len(y_te)} total ({y_te.mean():.1%} positive rate)")
print(f"AUC: {roc_auc_score(y_te, proba_te):.4f}")

In [ ]:
# Cost matrix: fraud detection scenario
COST = {
    "TN": 0,     # Correct reject: no cost
    "FP": -5,    # False alarm: analyst review time
    "FN": -500,  # Missed fraud: loss
    "TP": 50,    # Caught fraud: partial recovery minus investigation
}

def expected_profit(y_true, y_prob, threshold, cost):
    pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    tn, fp, fn, tp = cm.ravel()
    return (tn * cost["TN"] + fp * cost["FP"] +
            fn * cost["FN"] + tp * cost["TP"])

# Sweep thresholds
thresholds = np.linspace(0.01, 0.99, 200)
profits = [expected_profit(y_te, proba_te, t, COST) for t in thresholds]

optimal_idx = np.argmax(profits)
optimal_threshold = thresholds[optimal_idx]
optimal_profit    = profits[optimal_idx]

baseline_profit = expected_profit(y_te, proba_te, 0.5, COST)
print(f"Profit at threshold=0.50 : ${baseline_profit:,}")
print(f"Optimal threshold        : {optimal_threshold:.3f}")
print(f"Profit at optimal thresh : ${optimal_profit:,}")
print(f"Gain from threshold opt  : ${optimal_profit - baseline_profit:,}")

In [ ]:
def cost_weighted_confusion(y_true, y_prob, threshold, cost):
    pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    tn, fp, fn, tp = cm.ravel()
    cost_cm = np.array([
        [tn * cost["TN"], fp * cost["FP"]],
        [fn * cost["FN"], tp * cost["TP"]]
    ])
    labels = {
        "TN": (tn, cost["TN"]),
        "FP": (fp, cost["FP"]),
        "FN": (fn, cost["FN"]),
        "TP": (tp, cost["TP"]),
    }
    return cm, cost_cm, labels

cm_opt, cost_cm_opt, labels_opt = cost_weighted_confusion(y_te, proba_te, optimal_threshold, COST)
print(f"Confusion matrix at optimal threshold ({optimal_threshold:.3f}):")
print(pd.DataFrame(cm_opt, index=["True Neg","True Pos"], columns=["Pred Neg","Pred Pos"]))
print(f"\nCost-weighted confusion matrix ($):")
print(pd.DataFrame(cost_cm_opt, index=["True Neg","True Pos"], columns=["Pred Neg","Pred Pos"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Profit curve
axes[0].plot(thresholds, profits, color="steelblue", lw=2)
axes[0].axvline(optimal_threshold, color="red", linestyle="--", label=f"Optimal={optimal_threshold:.3f}")
axes[0].axvline(0.5, color="orange", linestyle=":", label="Default=0.5")
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_xlabel("Threshold"); axes[0].set_ylabel("Expected profit ($)")
axes[0].set_title("Profit curve"); axes[0].legend()

# ROC with operating points
fpr, tpr, roc_thresh = roc_curve(y_te, proba_te)
axes[1].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC={roc_auc_score(y_te,proba_te):.3f}")
# Mark optimal and default operating points
for t, label, colour in [(optimal_threshold, "Optimal", "red"), (0.5, "Default", "orange")]:
    pred = (proba_te >= t).astype(int)
    cm_t = confusion_matrix(y_te, pred)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    fpr_t = fp_t / (fp_t + tn_t); tpr_t = tp_t / (tp_t + fn_t)
    axes[1].scatter([fpr_t], [tpr_t], s=80, zorder=5, color=colour, label=label)
axes[1].plot([0,1],[0,1],"k--",lw=1)
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC with operating points"); axes[1].legend(fontsize=8)

# Cost matrix heatmap
import matplotlib.colors as mcolors
cost_display = np.array([[COST["TN"], COST["FP"]], [COST["FN"], COST["TP"]]], dtype=float)
im = axes[2].imshow(cost_display, cmap="RdYlGn")
axes[2].set_xticks([0,1]); axes[2].set_yticks([0,1])
axes[2].set_xticklabels(["Pred Neg","Pred Pos"])
axes[2].set_yticklabels(["True Neg","True Pos"])
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, f"${cost_display[i,j]:+.0f}", ha="center", va="center", fontsize=12, fontweight="bold")
plt.colorbar(im, ax=axes[2])
axes[2].set_title("Cost matrix ($)")

plt.tight_layout()
plt.savefig(f"{base}/14_cost_sensitive_eval.png", dpi=120)
plt.show()